# Paper-quality KnottedGraph vs Topoly scaling figure

This notebook is **plotting-only**: it does not rerun any Yamada calculation. It reads the sample-level benchmark CSV already produced by `03_knottedgraph_vs_topoly_scaling.ipynb`.

The three panels are:

- **(a) crossing scaling:** median runtime across all graph sizes at fixed projected crossing count $c$;
- **(b) vertex scaling:** arithmetic mean over **all benchmark rows having the same $V$**;
- **(c) edge scaling:** arithmetic mean over **all benchmark rows having the same $E$**.

Because the same crossing grid is used for each graph size, the $V$ and $E$ views average over the same crossing-complexity distribution at every size. If a calculation times out, the timeout threshold is used as a conservative lower bound in the size-average; such points are overplotted with an open marker.

Both a **3 rows × 1 column** and **1 row × 3 columns** publication layout are saved as PDF and 600-dpi PNG.

In [ ]:
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

RES = ROOT / "User_guide" / "benchmarks" / "results_latest"
FIG = ROOT / "User_guide" / "benchmarks" / "figures_latest"
FIG.mkdir(parents=True, exist_ok=True)

RAW_CSV = RES / "topoly_yamada_paper_scaling_raw.csv"
AGG_CSV = RES / "topoly_yamada_publication_aggregate.csv"

BOOTSTRAP_SAMPLES = 20_000
BOOTSTRAP_SEED = 20260818
FAMILY = "crossings_graph_ensemble"

# Styling chosen to match the clean, serif, panel-labelled visual language
# used in the paper figures.
KG_COLOR = "#0B6E69"
TOPOLY_COLOR = "#C2185B"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 10.5,
    "axes.titlesize": 11.5,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9.2,
    "ytick.labelsize": 9.2,
    "legend.fontsize": 9.0,
    "axes.linewidth": 0.9,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "savefig.transparent": False,
})

raw = pd.read_csv(RAW_CSV)
raw = raw.loc[raw["family"] == FAMILY].copy()

for col in ["V", "E", "crossings", "knottedgraph_s", "topoly_s", "timeout_s"]:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")

print(f"Loaded {len(raw)} sample rows")
print("crossing grid =", sorted(raw["crossings"].dropna().astype(int).unique()))
print("unique V =", raw["V"].nunique(), "; unique E =", raw["E"].nunique())


In [ ]:
def bootstrap_interval(values, *, statistic="mean", seed=0):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan, np.nan, np.nan

    stat = np.mean if statistic == "mean" else np.median
    center = float(stat(values))
    if values.size == 1:
        return center, center, center

    rng = np.random.default_rng(seed)
    idx = rng.integers(0, values.size, size=(BOOTSTRAP_SAMPLES, values.size))
    boot = stat(values[idx], axis=1)
    lo, hi = np.quantile(boot, [0.025, 0.975])
    return center, float(lo), float(hi)


def crossing_aggregate(df, framework):
    records = []
    for i, (c, group) in enumerate(df.groupby("crossings", sort=True)):
        ok = group.loc[group[f"{framework}_status"] == "ok", f"{framework}_s"].dropna().to_numpy()
        center, lo, hi = bootstrap_interval(
            ok,
            statistic="median",
            seed=BOOTSTRAP_SEED + 1009 * i + (framework == "topoly"),
        )
        records.append({
            "x": float(c),
            "estimate": center,
            "ci_low": lo,
            "ci_high": hi,
            "n_ok": int(len(ok)),
            "n_total": int(len(group)),
            "n_timeout": int((group[f"{framework}_status"] == "timeout").sum()),
        })
    return pd.DataFrame(records)


def size_aggregate(df, xkey, framework):
    """Average all rows sharing the same V/E, not a single crossing slice.

    Successful values enter exactly. A timeout contributes timeout_s, so any
    point with n_timeout>0 is a lower-bound estimate of the true arithmetic mean.
    Errors/skips are left missing rather than assigned an artificial runtime.
    """
    records = []
    for i, (x, group) in enumerate(df.groupby(xkey, sort=True)):
        values = []
        for _, row in group.iterrows():
            status = row[f"{framework}_status"]
            if status == "ok" and np.isfinite(row[f"{framework}_s"]):
                values.append(float(row[f"{framework}_s"]))
            elif status == "timeout" and np.isfinite(row["timeout_s"]):
                values.append(float(row["timeout_s"]))

        center, lo, hi = bootstrap_interval(
            values,
            statistic="mean",
            seed=BOOTSTRAP_SEED + 2003 * i + (framework == "topoly"),
        )
        records.append({
            "x": float(x),
            "estimate": center,
            "ci_low": lo,
            "ci_high": hi,
            "n_ok": int((group[f"{framework}_status"] == "ok").sum()),
            "n_total": int(len(group)),
            "n_timeout": int((group[f"{framework}_status"] == "timeout").sum()),
            "n_used": int(len(values)),
        })
    return pd.DataFrame(records)


aggregates = {}
for framework in ["knottedgraph", "topoly"]:
    aggregates[("crossings", framework)] = crossing_aggregate(raw, framework)
    aggregates[("V", framework)] = size_aggregate(raw, "V", framework)
    aggregates[("E", framework)] = size_aggregate(raw, "E", framework)

# Export the exact values used in the publication figure.
out = []
for (view, framework), frame in aggregates.items():
    tmp = frame.copy()
    tmp.insert(0, "framework", framework)
    tmp.insert(0, "view", view)
    tmp["statistic"] = "median" if view == "crossings" else "mean"
    out.append(tmp)
pd.concat(out, ignore_index=True).to_csv(AGG_CSV, index=False)
print("wrote", AGG_CSV)


In [ ]:
def style_axis(ax, *, xscale=None):
    ax.set_yscale("log")
    if xscale == "log2":
        ax.set_xscale("log", base=2)
    ax.grid(False)
    for side in ("top", "right", "bottom", "left"):
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("black")
        ax.spines[side].set_linewidth(0.9)
    ax.tick_params(which="both", top=False, right=False)


def draw_series(ax, frame, *, color, marker, linestyle, label):
    frame = frame[np.isfinite(frame["estimate"])].copy()
    if frame.empty:
        return

    x = frame["x"].to_numpy()
    y = frame["estimate"].to_numpy()
    lo = frame["ci_low"].to_numpy()
    hi = frame["ci_high"].to_numpy()

    ax.plot(
        x, y,
        color=color,
        marker=marker,
        linestyle=linestyle,
        linewidth=2.0,
        markersize=5.5,
        markeredgewidth=0.9,
        label=label,
        zorder=3,
    )
    ax.fill_between(x, lo, hi, color=color, alpha=0.13, linewidth=0, zorder=1)

    censored = frame[frame["n_timeout"] > 0]
    if not censored.empty:
        ax.scatter(
            censored["x"], censored["estimate"],
            marker=marker,
            s=48,
            facecolors="white",
            edgecolors=color,
            linewidths=1.15,
            zorder=4,
        )


def draw_panel(ax, view, panel_label):
    specs = {
        "crossings": ("Projected crossings, $c$", "Projected-crossing scaling", "log2"),
        "V": ("Graph vertices, $V$", "Vertex scaling", None),
        "E": ("Graph edges, $E$", "Edge scaling", None),
    }
    xlabel, title, xscale = specs[view]

    draw_series(
        ax, aggregates[(view, "knottedgraph")],
        color=KG_COLOR, marker="o", linestyle="-", label="KnottedGraph",
    )
    draw_series(
        ax, aggregates[(view, "topoly")],
        color=TOPOLY_COLOR, marker="s", linestyle="--", label="Topoly",
    )

    style_axis(ax, xscale=xscale)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Yamada evaluation time (s)")
    ax.set_title(title, pad=7)
    ax.text(
        -0.13, 1.04, panel_label,
        transform=ax.transAxes,
        fontsize=15,
        fontweight="bold",
        va="top",
        ha="left",
        clip_on=False,
    )


legend_handles = [
    Line2D([0], [0], color=KG_COLOR, marker="o", lw=2.0, label="KnottedGraph"),
    Line2D([0], [0], color=TOPOLY_COLOR, marker="s", lw=2.0, ls="--", label="Topoly"),
]


In [ ]:
# --- 3 rows x 1 column ---
fig, axes = plt.subplots(3, 1, figsize=(6.7, 12.0), constrained_layout=False)
for ax, view, label in zip(axes, ["crossings", "V", "E"], ["(a)", "(b)", "(c)"]):
    draw_panel(ax, view, label)

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.53, 0.995),
    ncol=2,
    frameon=False,
    handlelength=2.8,
    columnspacing=2.0,
)
fig.subplots_adjust(left=0.16, right=0.98, top=0.95, bottom=0.065, hspace=0.38)

vertical_pdf = FIG / "topoly_vs_knottedgraph_scaling_3x1.pdf"
vertical_png = FIG / "topoly_vs_knottedgraph_scaling_3x1.png"
fig.savefig(vertical_pdf, bbox_inches="tight")
fig.savefig(vertical_png, dpi=600, bbox_inches="tight")
plt.show()
print(vertical_pdf)
print(vertical_png)


In [ ]:
# --- 1 row x 3 columns ---
fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.25), constrained_layout=False)
for ax, view, label in zip(axes, ["crossings", "V", "E"], ["(a)", "(b)", "(c)"]):
    draw_panel(ax, view, label)

# One shared y label is visually cleaner in the horizontal paper layout.
for ax in axes[1:]:
    ax.set_ylabel("")

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.52, 1.02),
    ncol=2,
    frameon=False,
    handlelength=2.8,
    columnspacing=2.0,
)
fig.subplots_adjust(left=0.065, right=0.995, top=0.82, bottom=0.20, wspace=0.28)

horizontal_pdf = FIG / "topoly_vs_knottedgraph_scaling_1x3.pdf"
horizontal_png = FIG / "topoly_vs_knottedgraph_scaling_1x3.png"
fig.savefig(horizontal_pdf, bbox_inches="tight")
fig.savefig(horizontal_png, dpi=600, bbox_inches="tight")
plt.show()
print(horizontal_pdf)
print(horizontal_png)


### Interpretation of the size panels

The $V$ and $E$ panels are intentionally aggregated over all repeated rows with the same size. They are therefore **not** fixed-crossing slices.

For the current connected trivalent ensemble, $E=3V/2$. Consequently the vertex and edge panels are reparameterizations of the same size dependence; they should not be described as independent statistical evidence. They are both provided because $V$- and $E$-scaling are useful interfaces for different readers.

Open markers indicate points containing at least one timeout. In those size averages, each timeout is entered at the timeout threshold, so the plotted arithmetic mean is a conservative **lower bound** on the true mean runtime.